In [1]:
import pandas as pd
import numpy as np
import regex as re
from tqdm import tqdm
from collections import defaultdict
from mlxtend.preprocessing import TransactionEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [2]:
df = pd.read_csv(r'D:\Download\doc\SK_WIDIASIH\dataset_kompas\databersih\data_lokasiSMOTE_finalFPG.csv')

In [3]:
df.head()

,judul_berita,isi_berita,text_clean,label_kriminal,label_krim_gabungan,jenis_kriminal,prediksi_jenis_dt,motif_kejahatan,prediksi_motif_dt,hubungan_pelaku_korban,prediksi_hubungan_dt,karakteristik_kejahatan,prediksi_karakteristik_dt,waktu,prediksi_waktu_dt,lokasi,prediksi_lokasi_dt
0,Bocah yang Jasadnya Terbungkus Sarung Tewas Di...,"JAKARTA, KOMPAS.com - Pasangan pengemis bernam...",pasangan pengemis bernama sinta dewi 22 aidil ...,True,1,Pembunuhan,Pembunuhan,"Dendam, Emosi & Konflik","Dendam, Emosi & Konflik",Saling Kenal,Saling Kenal,Impulsif/Spontan,Impulsif/Spontan,Malam/Dini Hari,Malam/Dini Hari,Area Komersial,Area Komersial
1,Orangtua Bocah Tewas dalam Sarung Sering Aniay...,"JAKARTA, KOMPAS.com - Sinta Dewi (22) dan Aidi...",sinta dewi 22 aidil zacky rahman 19 pengemis d...,True,1,Penganiayaan & Kekerasan,Penganiayaan & Kekerasan,"Dendam, Emosi & Konflik","Dendam, Emosi & Konflik",Tidak Kenal/Tidak Diketahui,Tidak Kenal/Tidak Diketahui,Premeditasi/Terencana,Premeditasi/Terencana,Malam/Dini Hari,Malam/Dini Hari,Area Komersial,Area Komersial
2,Orangtua Bocah Tewas dalam Sarung Aniaya Anakn...,"JAKARTA, KOMPAS.com - Dua pengemis bernama Sin...",pengemis bernama sinta dewi 22 aidil zacky rah...,True,1,Pembunuhan,Penganiayaan & Kekerasan,"Dendam, Emosi & Konflik","Dendam, Emosi & Konflik",Tidak Kenal/Tidak Diketahui,Tidak Kenal/Tidak Diketahui,Impulsif/Spontan,Impulsif/Spontan,Malam/Dini Hari,Malam/Dini Hari,Area Komersial,Area Komersial
3,Orangtua Bocah Tewas Terbungkus Sarung di Beka...,"JAKARTA, KOMPAS.com - Balita berinisial RMR (3...",balita berinisial rmr 3 ditemukan tewas terbun...,True,1,Penganiayaan & Kekerasan,Penganiayaan & Kekerasan,"Dendam, Emosi & Konflik","Dendam, Emosi & Konflik",Saling Kenal,Saling Kenal,Impulsif/Spontan,Impulsif/Spontan,Malam/Dini Hari,Malam/Dini Hari,Area Komersial,Area Komersial
4,"Motif Orangtua Bunuh Anak di Bekasi, Emosi kar...","JAKARTA, KOMPAS.com - Polisi mengungkapkan, or...",polisi orangtua bernama sinta dewi 22 aidil za...,True,1,Pembunuhan,Pembunuhan,"Dendam, Emosi & Konflik","Dendam, Emosi & Konflik",Tidak Kenal/Tidak Diketahui,Tidak Kenal/Tidak Diketahui,Impulsif/Spontan,Impulsif/Spontan,Tidak Teridentifikasi,Tidak Teridentifikasi,Area Komersial,Area Komersial


# Data Cek

In [4]:
print("Jumlah baris data:")
jumlah_baris = len(df)
print(f"   Total baris data: {jumlah_baris}")

Jumlah baris data:
   Total baris data: 59141


In [5]:
jumlah_duplikasi = df.duplicated().sum()
print("Cek data duplikasi:")
print(f"   Jumlah baris duplikat: {jumlah_duplikasi}")

Cek data duplikasi:
   Jumlah baris duplikat: 0


In [6]:
print("Cek data kosong (Missing Values):")
data_kosong = df.isnull().sum()
print(data_kosong)

Cek data kosong (Missing Values):
judul_berita                 0
isi_berita                   0
text_clean                   0
label_kriminal               0
label_krim_gabungan          0
jenis_kriminal               0
prediksi_jenis_dt            0
motif_kejahatan              0
prediksi_motif_dt            0
hubungan_pelaku_korban       0
prediksi_hubungan_dt         0
karakteristik_kejahatan      0
prediksi_karakteristik_dt    0
waktu                        0
prediksi_waktu_dt            0
lokasi                       0
prediksi_lokasi_dt           0
dtype: int64


# Fp-Growth

In [7]:
cols_dt = [
    'prediksi_jenis_dt',
    'prediksi_motif_dt',
    'prediksi_hubungan_dt',
    'prediksi_karakteristik_dt',
    'prediksi_waktu_dt',
    'prediksi_lokasi_dt',
]
df_fpg = df[cols_dt].copy()

for col in cols_dt:
  clean_name = col.replace('prediksi_', '').replace('_dt', '')
  df_fpg[col] = clean_name + ': ' + df_fpg[col].astype(str)

df_encoded = pd.get_dummies(df_fpg, prefix='', prefix_sep='', dtype=int)
print('Data siap! Ukuran matriks:', df_encoded.shape)
df_encoded.head()

Data siap! Ukuran matriks: (59141, 26)


,jenis: Kejahatan Seksual dan Asusila,jenis: Kejahatan Siber,jenis: Narkotika,jenis: Pembunuhan,jenis: Pencurian & Perampokan,jenis: Penganiayaan & Kekerasan,jenis: Penipuan & Ekonomi,motif: Asmara & Kejahatan Seksual,"motif: Dendam, Emosi & Konflik",motif: Ekonomi & Harta Benda,...,waktu: Pagi Hari,waktu: Siang/Sore,waktu: Tidak Teridentifikasi,lokasi: Area Institusi/Kerja,lokasi: Area Komersial,lokasi: Area Privat (Hunian),lokasi: Area Publik (Umum),lokasi: Area Terbuka/Alam,lokasi: Ruang Digital/Siber,lokasi: Tidak Teridentifikasi
0,0,0,0,1,0,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
4,0,0,0,1,0,0,0,0,1,0,...,0,0,1,0,1,0,0,0,0,0


In [8]:
support_candidates = [0.001, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20]
summary_support = []

for sup in support_candidates:
    itemsets = fpgrowth(df_encoded, min_support=sup, use_colnames=True)

    itemsets_len_6 = itemsets[itemsets['itemsets'].apply(lambda x: len(x) == 6)]
    summary_support.append({
        'Min Support': sup,
        'Persentase Data': f'{sup*100:g}%',
        'Total Frequent Itemsets': len(itemsets),
        'Jumlah Pola 6 Variabel': len(itemsets_len_6)  
    })

df_summary = pd.DataFrame(summary_support)
display(df_summary)

C:\Users\Java Computer\anaconda3\envs\tensorflow\lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
C:\Users\Java Computer\anaconda3\envs\tensorflow\lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
C:\Users\Java Computer\anaconda3\envs\tensorflow\lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
C:\Users\Java Computer\anaconda3\envs\tensorflow\lib\site-packages\

,Min Support,Persentase Data,Total Frequent Itemsets,Jumlah Pola 6 Variabel
0,0.001,0.1%,4669,212
1,0.010,1%,1258,9
2,0.020,2%,666,2
3,0.030,3%,440,1
4,0.050,5%,249,0
5,0.080,8%,124,0
6,0.100,10%,85,0
7,0.150,15%,41,0
8,0.200,20%,21,0


# Percobaan Labeling Manual

In [9]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

frequent_itemsets = fpgrowth(df_encoded, min_support=0.01, use_colnames=True)
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.85)

rules_filtered = rules[rules['lift'] > 2].copy()

rules_filtered['jumlah_variabel'] = rules_filtered.apply( lambda r: len(r['antecedents'] | r['consequents']), axis=1)

rules_flexible = rules_filtered[rules_filtered['jumlah_variabel'] >= 2].copy()

rules_flexible['antecedents_str'] = rules_flexible['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_flexible['consequents_str'] = rules_flexible['consequents'].apply(lambda x: ', '.join(list(x)))

rules_flexible = rules_flexible.sort_values(by=['jumlah_variabel', 'confidence', 'lift'],ascending=[False, False, False],)

print( f' Ditemukan {len(rules_flexible)} aturan asosiasi).\n')
display(
    rules_flexible[[
        'jumlah_variabel',
        'antecedents_str',
        'consequents_str',
        'support',
        'confidence',
        'lift',
    ]]
)

C:\Users\Java Computer\anaconda3\envs\tensorflow\lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


 Ditemukan 134 aturan asosiasi).



,jumlah_variabel,antecedents_str,consequents_str,support,confidence,lift
341,6,"jenis: Penipuan & Ekonomi, hubungan: Tidak Kenal/Tidak Diketahui, karakteristik: Premeditasi/Terencana, lokasi: Area Institusi/Kerja, waktu: Tidak Teridentifikasi",motif: Ekonomi & Harta Benda,0.035221,0.993324,2.080910
351,6,"jenis: Penipuan & Ekonomi, karakteristik: Premeditasi/Terencana, lokasi: Area Institusi/Kerja, waktu: Tidak Teridentifikasi, hubungan: Saling Kenal",motif: Ekonomi & Harta Benda,0.012665,0.992053,2.078248
384,6,"lokasi: Ruang Digital/Siber, jenis: Penipuan & Ekonomi, hubungan: Tidak Kenal/Tidak Diketahui, karakteristik: Premeditasi/Terencana, waktu: Tidak Teridentifikasi",motif: Ekonomi & Harta Benda,0.013070,0.985969,2.065503
254,6,"jenis: Penipuan & Ekonomi, hubungan: Tidak Kenal/Tidak Diketahui, karakteristik: Premeditasi/Terencana, waktu: Tidak Teridentifikasi, lokasi: Tidak Teridentifikasi",motif: Ekonomi & Harta Benda,0.010264,0.983793,2.060943
63,6,"jenis: Penipuan & Ekonomi, hubungan: Tidak Kenal/Tidak Diketahui, karakteristik: Premeditasi/Terencana, waktu: Tidak Teridentifikasi, lokasi: Area Komersial",motif: Ekonomi & Harta Benda,0.016097,0.982456,2.058143
342,6,"jenis: Penipuan & Ekonomi, hubungan: Tidak Kenal/Tidak Diketahui, lokasi: Area Institusi/Kerja, waktu: Tidak Teridentifikasi","motif: Ekonomi & Harta Benda, karakteristik: Premeditasi/Terencana",0.035221,0.981159,2.211637
352,6,"jenis: Penipuan & Ekonomi, hubungan: Saling Kenal, lokasi: Area Institusi/Kerja, waktu: Tidak Teridentifikasi","motif: Ekonomi & Harta Benda, karakteristik: Premeditasi/Terencana",0.012665,0.973992,2.195482
255,6,"jenis: Penipuan & Ekonomi, hubungan: Tidak Kenal/Tidak Diketahui, lokasi: Tidak Teridentifikasi, waktu: Tidak Teridentifikasi","motif: Ekonomi & Harta Benda, karakteristik: Premeditasi/Terencana",0.010264,0.972756,2.192697
385,6,"jenis: Penipuan & Ekonomi, hubungan: Tidak Kenal/Tidak Diketahui, lokasi: Ruang Digital/Siber, waktu: Tidak Teridentifikasi","motif: Ekonomi & Harta Benda, karakteristik: Premeditasi/Terencana",0.013070,0.972327,2.191729
64,6,"jenis: Penipuan & Ekonomi, hubungan: Tidak Kenal/Tidak Diketahui, lokasi: Area Komersial, waktu: Tidak Teridentifikasi","motif: Ekonomi & Harta Benda, karakteristik: Premeditasi/Terencana",0.016097,0.969450,2.185244
